# Lección 10.3 — Pooling y jerarquía de características

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 10.2 (convoluciones), Lección 9.6 (hiperparámetros de la red)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Calcular a mano un **max pooling** y un **average pooling**, y explicar para qué sirven.
- Explicar la **jerarquía de características**: bordes → formas → partes → objetos, y el **campo receptivo**.
- Construir y entrenar tu **primera CNN completa** en PyTorch y compararla con una red densa del mismo tamaño.
- Mirar "dentro" de la red: los filtros aprendidos y los mapas de características.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación
Cargamos MNIST (Lección 10.1) y una función de entrenamiento como la de la Lección 9.6. No hace falta entenderla línea por línea.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets

plt.rcParams["figure.figsize"] = (8, 4.5)
device = "cuda" if torch.cuda.is_available() else "cpu"

mnist = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)
X_tr = mnist.data[:10000].float().div(255).unsqueeze(1)       # 10,000 dígitos → forma (10000, 1, 28, 28)
y_tr = mnist.targets[:10000]
X_te = mnist_test.data[:2000].float().div(255).unsqueeze(1)   # 2,000 para prueba
y_te = mnist_test.targets[:2000]

def entrenar(modelo, epocas=3, lr=1e-3):
    """Entrena con Adam y entropía cruzada (Lecciones 9.5 y 3.2), en lotes de 64."""
    torch.manual_seed(0)
    modelo.to(device).train()
    opt, perdida_fn = torch.optim.Adam(modelo.parameters(), lr=lr), nn.CrossEntropyLoss()
    for _ in range(epocas):
        orden = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 64):
            idx = orden[i:i + 64]
            perdida = perdida_fn(modelo(X_tr[idx].to(device)), y_tr[idx].to(device))
            opt.zero_grad(); perdida.backward(); opt.step()
    return modelo.eval()

def exactitud(modelo, X, y=y_te):
    with torch.no_grad():
        return (modelo(X.to(device)).argmax(1).cpu() == y).float().mean().item()

contar = lambda m: sum(p.numel() for p in m.parameters())
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: resumir un mapa

Imagina que ves un mapa de tu ciudad con **todas** las calles y casas. Para planear un viaje en auto no necesitas ese detalle: te basta con un mapa más pequeño que diga, por cada barrio, *"¿hay una autopista aquí?"*.

El **pooling** (agrupamiento) hace eso con los mapas de características de la Lección 10.2:
1. Divide el mapa en **cuadritos** (normalmente de 2 × 2).
2. De cada cuadrito se queda con **un solo número**: el **máximo** (*max pooling*) o el **promedio** (*average pooling*).

```
Mapa de 4×4                      Max pooling 2×2
┌───┬───┬───┬───┐
│ 1 │ 3 │ 0 │ 2 │                 ┌───┬───┐
│ 5 │ 2 │ 1 │ 0 │      ──►        │ 5 │ 2 │
├───┼───┼───┼───┤                 ├───┼───┤
│ 0 │ 0 │ 7 │ 1 │                 │ 4 │ 7 │
│ 4 │ 1 │ 3 │ 2 │                 └───┴───┘
└───┴───┴───┴───┘
```

El max pooling responde a la pregunta: *"¿apareció mi patrón **en algún lugar** de este cuadrito?"*. No importa exactamente dónde.

### ✋ Ejercicio 1 — Pooling a mano
Con el mismo mapa de 4 × 4 de arriba, calcula a mano el resultado de un **average pooling** 2 × 2 (el promedio de cada cuadrito).

✍️ *Mi resultado:*
```
┌─────┬─────┐
│     │     │
├─────┼─────┤
│     │     │
└─────┴─────┘
```

In [ ]:
mapa = np.array([[1, 3, 0, 2],
                 [5, 2, 1, 0],
                 [0, 0, 7, 1],
                 [4, 1, 3, 2]])

def pooling(m, tam=2, modo="max"):
    """Pooling a mano: parte el mapa en cuadritos de tam×tam y resume cada uno."""
    alto, ancho = m.shape[0] // tam, m.shape[1] // tam
    cuadritos = m[:alto * tam, :ancho * tam].reshape(alto, tam, ancho, tam)
    return cuadritos.max(axis=(1, 3)) if modo == "max" else cuadritos.mean(axis=(1, 3))

print("Max pooling:\n", pooling(mapa, modo="max"))
print("Average pooling:\n", pooling(mapa, modo="promedio"))

<details><summary><b>👀 Ver respuesta</b></summary>

Average pooling: `[[2.75, 0.75], [1.25, 3.25]]`. Por ejemplo, el primer cuadrito: (1 + 3 + 5 + 2) / 4 = 2.75.

Fíjate en la diferencia: el **max** conserva la señal más fuerte ("aquí **sí** hay un borde"), mientras que el **promedio** la diluye con los ceros de alrededor. Por eso dentro de las CNN se usa casi siempre **max pooling**.
</details>

---
## 2. 📖 El concepto: ¿para qué sirve el pooling?

1. **Reduce el tamaño** del mapa: un pooling 2 × 2 deja la cuarta parte de los números. Menos cómputo y menos memoria en las capas siguientes.
2. **Da tolerancia a pequeños desplazamientos**: si el patrón se mueve un píxel pero sigue dentro del mismo cuadrito, el máximo **no cambia**.
3. **Amplía la "vista"** de las capas siguientes (lo verás en la Sección 3).

Y un detalle importante: **el pooling no tiene parámetros que aprender**. Es una operación fija.

### 🔮 Ejercicio 2 — Muévete un poquito
Vamos a mover el contenido del mapa **un píxel a la derecha** (el 7 pasa de la columna 2 a la 3, etc.). ¿Cambiará el resultado del **max pooling**? ¿Cambiará el **mapa** original?

✍️ *Mi predicción:*

In [ ]:
movido = np.roll(mapa, 1, axis=1)
print("Mapa movido 1 px:\n", movido)
print("\nMax pooling del original:\n", pooling(mapa))
print("Max pooling del movido:\n", pooling(movido))
print(f"\nNúmeros del mapa que cambiaron        : {(mapa != movido).sum()} de {mapa.size}")
print(f"Números del max pooling que cambiaron : {(pooling(mapa) != pooling(movido)).sum()} de {pooling(mapa).size}")

<details><summary><b>👀 Ver respuesta</b></summary>

El mapa cambió en **15 de 16** posiciones, pero el max pooling solo cambió en **1 de 4**: el 5 y el 7 siguen dentro de su cuadrito, así que siguen ganando. Solo cambia el cuadrito de arriba a la derecha, al que "entró" el 3 desde la izquierda.

El pooling **no** da una tolerancia perfecta (si el patrón cruza al cuadrito vecino, cambia), pero al apilar varios poolings la red se vuelve bastante tolerante a pequeños movimientos. Lo comprobarás con dígitos de verdad en la Sección 4.
</details>

### 📖 Pooling en PyTorch
`nn.MaxPool2d(2)` hace exactamente lo mismo, en cada canal por separado:

In [ ]:
capa_pool = nn.MaxPool2d(kernel_size=2)
mapas = torch.rand(1, 16, 28, 28)                     # 16 mapas de características de 28×28
print("Antes del pooling :", tuple(mapas.shape))
print("Después           :", tuple(capa_pool(mapas).shape))
print("Parámetros del pooling:", contar(capa_pool))

### ✋ Ejercicio 3
1. ¿Por qué el número de **canales** (16) no cambia con el pooling?
2. Si aplicas `MaxPool2d(2)` **tres veces** seguidas a un mapa de 28 × 28, ¿qué tamaño queda?

<details><summary><b>👀 Ver respuesta</b></summary>

1. Porque el pooling resume **cada mapa por separado**: no mezcla canales. Solo reduce alto y ancho.
2. 28 → 14 → 7 → **3** (7 / 2 = 3.5, se redondea hacia abajo y se pierde la última fila y columna). Con imágenes pequeñas, demasiado pooling deja mapas diminutos: es uno de los errores comunes de la Sección 6.
</details>

---
## 3. 📖 Cómo funciona por dentro: la jerarquía de características

Una CNN apila bloques **convolución → ReLU → pooling**. Cada bloque trabaja sobre la salida del anterior:

```
Imagen ──► [Conv → ReLU → Pool] ──► [Conv → ReLU → Pool] ──► [Conv → ReLU → Pool] ──► Densa ──► "gato"
              bordes, colores          esquinas, curvas,         ojos, orejas,
                                       texturas                  ruedas, letras
```

Es como leer: primero distingues **trazos**, con los trazos formas **letras**, con las letras **palabras** y con las palabras **frases**. Cada capa combina los patrones de la anterior en algo más complejo.

### 📖 El campo receptivo: cuánto "ve" cada neurona
Una neurona de la primera capa (kernel 3 × 3) solo ve **3 × 3 píxeles**. Pero una neurona de la segunda capa mira 3 × 3 posiciones del mapa anterior, y **cada** una de ellas ya veía 3 × 3 píxeles: en total ve **5 × 5** píxeles de la imagen original. Esa zona es su **campo receptivo**.

El pooling acelera el crecimiento: después de un pooling 2 × 2, cada paso en el mapa equivale a **2** píxeles de la imagen.

| Capa | Campo receptivo | Cada paso del mapa equivale a… |
|---|---|---|
| Conv 3×3 | 3 × 3 | 1 px |
| + Conv 3×3 | 5 × 5 | 1 px |
| + MaxPool 2×2 | 6 × 6 | 2 px |
| + Conv 3×3 | 6 + 2·2 = 10 × 10 | 2 px |

Regla: cada conv con kernel *k* suma **(k − 1) × (tamaño del paso actual)**; cada pooling 2 × 2 suma 1 × paso y **duplica** el paso.

### ✋ Ejercicio 4 — Calcula el campo receptivo
La CNN que vas a entrenar en un momento tiene esta secuencia:

`Conv 3×3 → MaxPool 2×2 → Conv 3×3 → MaxPool 2×2`

¿Qué campo receptivo tiene una neurona después de la **segunda conv**? ¿Y después del **segundo pooling**? (La imagen mide 28 × 28: ¿ve la neurona el dígito entero?)

✍️ *Mis cuentas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Conv 3×3: **3**, paso 1.
- MaxPool: 3 + 1 = **4**, el paso pasa a 2.
- Conv 3×3: 4 + 2 × 2 = **8**, paso 2.
- MaxPool: 8 + 1 × 2 = **10**, el paso pasa a 4.

Así que cada neurona del final "ve" un trozo de **10 × 10** píxeles: un buen pedazo del trazo, pero no el dígito entero de 28 × 28. La **capa densa** del final es la que junta todas esas vistas parciales para decidir. Las redes profundas (Lección 10.4) apilan muchas más capas para que las últimas neuronas vean el objeto completo.
</details>

---
## 4. 📖 Tu primera CNN completa

Ya tienes todas las piezas. Esta es una CNN pequeña para MNIST:

In [ ]:
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # (1, 28, 28)  → (16, 28, 28): 16 detectores de trazos
    nn.ReLU(),
    nn.MaxPool2d(2),                              # (16, 28, 28) → (16, 14, 14)
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # (16, 14, 14) → (32, 14, 14): 32 detectores de formas
    nn.ReLU(),
    nn.MaxPool2d(2),                              # (32, 14, 14) → (32, 7, 7)
    nn.Flatten(),                                 # (32, 7, 7)   → 1,568 números en fila
    nn.Linear(32 * 7 * 7, 10),                    # → 10 puntuaciones, una por dígito
)

# Truco útil: pasa un "dígito" de ceros y mira la forma después de cada capa
x = torch.zeros(1, 1, 28, 28)
for capa in cnn:
    x = capa(x)
    print(f"{capa.__class__.__name__:<10} → {str(tuple(x.shape)):<18} parámetros: {contar(capa):>6,}")
print(f"\nTotal: {contar(cnn):,} parámetros")

Para que la comparación sea justa, construimos una **red densa** (Lección 9.2) con casi el **mismo número de parámetros**: aplana el dígito y usa una capa oculta de 26 neuronas.

### 🔮 Ejercicio 5 — CNN vs. red densa
Las dos redes tienen ≈20,500 parámetros y se entrenan igual (3 épocas, 10,000 dígitos).
1. ¿Cuál tendrá mejor exactitud en prueba?
2. ¿Cuál resistirá mejor que movamos los dígitos 1–4 píxeles (como en la Lección 10.1)?

✍️ *Mis predicciones:*

▶️ La celda tarda ≈30 segundos en CPU (la CNN hace muchas más cuentas que la densa, aunque tenga los mismos parámetros).

In [ ]:
torch.manual_seed(0)
densa = nn.Sequential(nn.Flatten(), nn.Linear(784, 26), nn.ReLU(), nn.Linear(26, 10))
print(f"Parámetros → densa: {contar(densa):,} · CNN: {contar(cnn):,}\n")

entrenar(densa); entrenar(cnn)

print("Desplazamiento →  densa     CNN")
for px in range(5):                                   # ✏️ prueba también desplazamientos hacia abajo: dims=2
    movidos = torch.roll(X_te, px, dims=3)
    print(f"      {px} px     →  {exactitud(densa, movidos):5.1%}   {exactitud(cnn, movidos):5.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. La **CNN** gana claramente sin desplazamiento: ≈95 % contra ≈88 %, con el mismo número de parámetros. Los usa mejor porque cada filtro se aplica en todas las posiciones.
2. La **CNN** también resiste mucho mejor los desplazamientos: con 2 px conserva ≈85 % (la densa cae a ≈63 %) y con 3 px ≈67 % (la densa ≈44 %).

Ojo: la CNN **tampoco** es inmune. Con 4 px también cae bastante (≈42 %, contra ≈26 % de la densa), porque en el entrenamiento todos los dígitos estaban centrados. La convolución y el pooling ayudan; para una tolerancia real hay que **mostrarle** dígitos movidos al entrenar (*data augmentation*, Lección 9.8).
</details>

### 📖 Mirar dentro de la red (1): los filtros aprendidos
Los 16 filtros de la primera capa empezaron al azar. Veamos qué aprendieron:

In [ ]:
filtros = cnn[0].weight.detach().cpu()[:, 0]          # 16 filtros de 3×3
fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
for a, f in zip(ax.flat, filtros):
    a.imshow(f, cmap="RdBu_r", vmin=-filtros.abs().max(), vmax=filtros.abs().max()); a.axis("off")
plt.suptitle("Filtros de la primera capa (rojo = positivo, azul = negativo)"); plt.show()

### ✋ Ejercicio 6
¿Reconoces en alguno de los filtros un patrón parecido a los de la Lección 10.2 (un lado positivo y el opuesto negativo)? ¿Qué crees que detecta?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Normalmente verás varios filtros con una **mitad roja y otra azul**: son **detectores de bordes** en distintas direcciones (vertical, horizontal, diagonal), parecidos a Sobel. Otros tienen una zona clara rodeada de oscuro: detectan **trazos** o puntos. Nadie se los enseñó: la red los "inventó" porque le sirven para distinguir dígitos. (Con filtros de 3 × 3 los patrones son toscos; en redes grandes entrenadas con fotos, los filtros de la primera capa se parecen muchísimo a detectores de bordes y de colores.)
</details>

### 📖 Mirar dentro de la red (2): los mapas de características
Ahora pasamos un dígito por la red y miramos qué sale **después de cada bloque**.

In [ ]:
digito = X_te[0:1].to(device)
with torch.no_grad():
    mapas_1 = cnn[:3](digito)[0].cpu()        # tras el bloque 1: (16, 14, 14)
    mapas_2 = cnn[:6](digito)[0].cpu()        # tras el bloque 2: (32, 7, 7)

fig, ax = plt.subplots(3, 8, figsize=(13, 5))
ax[0, 0].imshow(X_te[0, 0], cmap="gray"); ax[0, 0].set_title(f"Entrada {tuple(X_te[0, 0].shape)}", fontsize=8)
for i in range(8):
    ax[1, i].imshow(mapas_1[i], cmap="viridis")
    ax[2, i].imshow(mapas_2[i], cmap="viridis")
ax[1, 0].set_title(f"Bloque 1: 8 de 16 mapas {tuple(mapas_1.shape[1:])}", fontsize=8, loc="left")
ax[2, 0].set_title(f"Bloque 2: 8 de 32 mapas {tuple(mapas_2.shape[1:])}", fontsize=8, loc="left")
for a in ax.flat: a.axis("off")
plt.show()

### 🔮 Ejercicio 7 — ¿Qué ves en cada nivel?
Compara las filas 2 y 3. ¿En cuál todavía reconoces la forma del dígito? ¿Por qué en la fila 3 ya es más difícil?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En el **bloque 1** (14 × 14) todavía se reconoce el 7: cada mapa resalta partes distintas del trazo (bordes de un lado, del otro, el trazo completo…). En el **bloque 2** (7 × 7) los mapas son más pequeños y **abstractos**: cada uno dice "aquí hay **tal combinación** de trazos" (una esquina, el final de una línea…), y para un humano ya no se parece al dígito. Es la **jerarquía**: cuanto más profunda la capa, menos "píxeles" y más "conceptos".
</details>

---
## 5. 📖 Hiperparámetros del pooling y de la arquitectura

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `kernel_size` del pooling | Tamaño del cuadrito que se resume | Reduce más rápido: más tolerancia, pero pierde detalle de posición | Reduce menos | **2** (con `stride=2`) |
| Tipo de pooling | Cómo se resume el cuadrito | — | — | **Max** entre capas; **promedio global** al final |
| Pooling global (`AdaptiveAvgPool2d(1)`) | Resume **todo** el mapa en un solo número por canal | — | — | Al final de redes modernas (ResNet), en vez de `Flatten` |
| Número de bloques conv+pool | Profundidad y campo receptivo | Ve patrones más grandes y abstractos; más lenta | Más simple; puede no ver el objeto completo | 2–3 para imágenes pequeñas (28–32 px) |
| Filtros por capa | Cuántos patrones busca cada nivel | Más capacidad (y más riesgo de sobreajuste) | Menos capacidad | 16–32 y **duplicar** tras cada pooling |

> 💡 **La regla de oro del diseño clásico:** cada vez que el pooling divide el tamaño entre 2, se **duplica** el número de filtros (16 → 32 → 64…). Así cada capa tiene más o menos el mismo "trabajo", con menos posiciones pero más tipos de patrones.

### 🔮 Ejercicio 8 — El pooling global
`nn.AdaptiveAvgPool2d(1)` promedia cada mapa completo y deja **un número por canal**, sin importar el tamaño de la entrada. ¿Qué forma tendrá la salida para una entrada de `(1, 32, 7, 7)`? ¿Y para `(1, 32, 50, 80)`?

In [ ]:
global_pool = nn.AdaptiveAvgPool2d(1)
for forma in [(1, 32, 7, 7), (1, 32, 50, 80)]:      # ✏️ prueba otras formas
    print(forma, "→", tuple(global_pool(torch.rand(forma)).shape))

<details><summary><b>👀 Ver respuesta</b></summary>

En los dos casos: `(1, 32, 1, 1)`, es decir, **32 números**, uno por mapa. Esto tiene una ventaja enorme: la parte densa que viene después **no depende del tamaño de la imagen**. Por eso redes como ResNet (Lección 10.4) aceptan fotos de distintos tamaños. Con `Flatten` en cambio, el número de entradas de la capa densa cambia con el tamaño de la foto y habría que rediseñarla.
</details>

### ✏️ Ejercicio 9 — ¿Cuántos filtros?
Entrena la misma CNN con **menos** y **más** filtros y compara parámetros, exactitud y tiempo. Cambia `F1` y `F2` (por ejemplo 4 y 8, o 32 y 64). ▶️ Tarda entre 10 y 40 segundos según los valores.

In [ ]:
import time

F1, F2 = 4, 8          # ✏️ filtros del bloque 1 y del bloque 2 (la red de la Sección 4 usaba 16 y 32)

torch.manual_seed(0)
cnn_prueba = nn.Sequential(
    nn.Conv2d(1, F1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(F1, F2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(F2 * 7 * 7, 10))
inicio = time.time(); entrenar(cnn_prueba)
print(f"F1={F1}, F2={F2} → {contar(cnn_prueba):,} parámetros · exactitud {exactitud(cnn_prueba, X_te):.1%} · "
      f"{time.time() - inicio:.0f} s de entrenamiento")

<details><summary><b>👀 Ver respuesta</b></summary>

Resultados aproximados (3 épocas, 10,000 dígitos):

| F1, F2 | Parámetros | Exactitud |
|---|---|---|
| 4, 8 | ≈4,300 | ≈89 % |
| 16, 32 | ≈20,500 | ≈95 % |
| 32, 64 | ≈50,000 | ≈96 % |

Más filtros = más capacidad, pero con **rendimientos decrecientes**: de 4 a 16 filtros se gana mucho; de 16 a 32, poco, y el entrenamiento tarda más. Es el mismo compromiso de siempre entre capacidad y costo (Lección 9.6).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| *mat1 and mat2 shapes cannot be multiplied* al llegar a la capa densa | El número de entradas de `nn.Linear` no coincide con lo que sale de `Flatten` | Pasa un tensor de ceros y mira la forma (truco de la Sección 4), o usa pooling global |
| El mapa queda de 1 × 1 o de 0 × 0 | Demasiados poolings para una imagen pequeña | Menos poolings, o padding para no perder píxeles |
| La red no distingue **dónde** está algo (p. ej. "¿el gato está a la izquierda?") | El pooling tira información de posición a propósito | Menos pooling, o arquitecturas de detección/segmentación (Lección 10.6) |
| Exactitud buena en prueba, mala con objetos movidos o girados | Los datos de entrenamiento siempre estaban centrados | *Data augmentation* (Lección 9.8) |
| CNN muy lenta en CPU | Muchos filtros sobre mapas grandes (antes del pooling) | Reducir el tamaño pronto (stride/pooling) y usar pocos filtros al principio |

### 🔮 Ejercicio 10 — El error clásico
Esta red está pensada para fotos de **32 × 32** (como CIFAR-10) pero alguien copió la capa densa de la red de MNIST. ¿Dará error? ¿Qué número hay que poner en lugar de `32 * 7 * 7`?

In [ ]:
red_cifar = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 10))         # ✏️ arréglame
try:
    print(tuple(red_cifar(torch.zeros(1, 3, 32, 32)).shape))
except RuntimeError as e:
    print("❌ Error:", str(e)[:90])
print("Lo que sale de Flatten:", red_cifar[:7](torch.zeros(1, 3, 32, 32)).shape[1])

<details><summary><b>👀 Ver respuesta</b></summary>

Sí da error: con 32 × 32 los poolings dejan 32 → 16 → **8**, así que `Flatten` produce 32 × 8 × 8 = **2,048** números, no 32 × 7 × 7 = 1,568. Hay que poner `nn.Linear(32 * 8 * 8, 10)`. O mejor: usar `nn.AdaptiveAvgPool2d(1)` + `nn.Linear(32, 10)`, que funciona con cualquier tamaño.
</details>

---
## 7. 📋 Ficha del modelo: red neuronal convolucional (CNN)

| | Red convolucional (CNN) |
|---|---|
| **Tipo** | Supervisado (sobre todo clasificación de imágenes; también audio y series) |
| **Intuición** | Una pila de "lupas": las primeras detectan bordes, las siguientes combinan bordes en formas y partes; una capa densa final decide |
| **Bloque básico** | Conv → ReLU → Pooling (repetido), y al final Flatten o pooling global + capa(s) densa(s) |
| **Cómo aprende** | Descenso de gradiente y retropropagación sobre los filtros y la parte densa (como cualquier red, M9) |
| **Hiperparámetros clave** | Nº de bloques, filtros por capa, `kernel_size`, pooling, más los de cualquier red (lr, batch, épocas, regularización) |
| **✅ Fortalezas** | Pocos parámetros para su potencia · Tolera pequeños desplazamientos · Aprende sus propias características · Jerarquía natural de bordes → objetos |
| **❌ Debilidades** | Necesita muchos datos etiquetados (o *transfer learning*, 10.5) · Costosa sin GPU · No tolera rotaciones o cambios de escala grandes si no los vio · Difícil de interpretar |
| **Úsala cuando…** | Clasificas imágenes o cualquier dato con estructura de cuadrícula |
| **Evítala cuando…** | Tus datos son tabulares (usa ensambles de árboles, M6) o tienes muy pocas imágenes y no puedes usar un modelo preentrenado |

## 8. 🧠 Resumen en 5 puntos
1. El **pooling** resume cada cuadrito de un mapa en un número (máximo o promedio): reduce el tamaño, da tolerancia a pequeños movimientos y **no tiene parámetros**.
2. Una CNN apila bloques **conv → ReLU → pool**: cada capa combina los patrones de la anterior (**bordes → formas → partes → objetos**).
3. El **campo receptivo** crece con cada capa (y más rápido tras cada pooling): las capas profundas "ven" zonas más grandes.
4. Con los **mismos parámetros**, una CNN supera a una red densa en imágenes y resiste mejor los desplazamientos.
5. Regla práctica: tras cada pooling, **duplica los filtros**; y si dudas del tamaño antes de la capa densa, **pasa un tensor de ceros** o usa **pooling global**.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Cuántos parámetros aprende una capa `nn.MaxPool2d(2)`?
- a) 4 · b) 2 · c) 0 · d) Depende de los canales

**2.** Un mapa de `(64, 56, 56)` pasa por `MaxPool2d(2)`. Queda…
- a) `(32, 56, 56)` · b) `(64, 28, 28)` · c) `(32, 28, 28)` · d) `(64, 54, 54)`

**3.** ¿Por qué las capas profundas de una CNN detectan cosas más complejas que las primeras?
- a) Porque tienen kernels más grandes
- b) Porque combinan los patrones de las capas anteriores y tienen un campo receptivo mayor
- c) Porque usan average pooling
- d) Porque tienen más parámetros densos

**4.** Para que la red funcione con fotos de **cualquier tamaño**, ¿qué conviene poner antes de la capa densa final?
- a) Más capas de convolución · b) `nn.Flatten()` · c) `nn.AdaptiveAvgPool2d(1)` · d) Un kernel de 7 × 7

**5.** Max pooling vs. average pooling **entre** capas convolucionales: ¿por qué suele preferirse el max?
- a) Es más rápido de entrenar porque tiene menos parámetros
- b) Conserva la señal más fuerte ("el patrón apareció aquí") en lugar de diluirla
- c) Da mapas más grandes
- d) No hay diferencia

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** El pooling es una operación fija, sin pesos.
2. **b)** Los canales no cambian; alto y ancho se dividen entre 2.
3. **b)** Jerarquía de características + campo receptivo creciente.
4. **c)** El pooling global deja un número por canal, sea cual sea el tamaño.
5. **b)** Los dos tienen 0 parámetros; la diferencia es qué resumen.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** El pooling hace que la red "olvide" la posición exacta de las cosas. Para **clasificar** ("¿hay un gato?") eso es bueno. Da un ejemplo de tarea de visión en la que **perder la posición sea un problema**.

**B. Caso 💼** Una empresa de seguridad quiere detectar **personas** en cámaras de vigilancia de 1,920 × 1,080. Las personas pueden aparecer muy pequeñas (lejos) o enormes (cerca). Con lo que sabes de campo receptivo, ¿por qué una CNN de solo 2 bloques conv+pool tendría problemas con las personas **cercanas**? ¿Y qué le pasa a una persona muy lejana (≈10 × 20 px) si la red aplica 6 poolings?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Por ejemplo: **detección de objetos** (dibujar una caja alrededor de cada auto), **segmentación** (pintar exactamente qué píxeles son tumor en una resonancia), contar objetos, leer texto en orden o saber si el peatón está **en** el paso de cebra. Lo verás en la Lección 10.6: allí las redes conservan o "recuperan" la información de posición.

**B.** Con 2 bloques el campo receptivo final es de ≈10 × 10 píxeles: ninguna neurona "ve" a una persona cercana de 500 píxeles de alto completa; solo ve trozos de ropa o de piel, y la capa densa tendría que deducir la persona a partir de miles de trozos. Hacen falta muchas más capas. En el otro extremo, 6 poolings dividen el tamaño entre 2⁶ = 64: una persona de 10 × 20 px queda reducida a **menos de un píxel** del mapa final y su señal se mezcla con el fondo. Por eso los detectores modernos combinan mapas de **varias escalas** (unos con mucho detalle para objetos pequeños, otros muy profundos para objetos grandes).
</details>

### 9.3 🧩 Práctica integradora: "¿Me sirve el pooling?"

Vas a hacer un experimento controlado: la misma CNN **con** y **sin** pooling. Responde cada paso antes de mirar.

**Paso 1 🔮 — Predice el tamaño.** Sin pooling (y con `padding=1`), los mapas siguen siendo de 28 × 28 al final. ¿Cuántos parámetros tendrá la capa densa final con 32 filtros? Compáralo con los 15,690 de la CNN con pooling.

✍️ *Mi cuenta:*

In [ ]:
torch.manual_seed(0)
sin_pool = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
    nn.Flatten(), nn.Linear(32 * 28 * 28, 10))
print(f"Capa densa final  → con pooling: {contar(cnn[-1]):,} · sin pooling: {contar(sin_pool[-1]):,}")
print(f"Red completa      → con pooling: {contar(cnn):,} · sin pooling: {contar(sin_pool):,}")

<details><summary><b>👀 Ver respuesta</b></summary>

32 × 28 × 28 = 25,088 entradas × 10 salidas + 10 sesgos = **250,890** parámetros solo en la capa densa: **16 veces** más que con pooling (15,690). La red completa pasa de ≈20,500 a ≈255,700 parámetros. El pooling es, también, una forma de **ahorrar parámetros**.
</details>

**Paso 2 🔮 — ¿Quién gana?** Ahora entrenamos la red sin pooling igual que la otra (▶️ tarda ≈1 min). Antes de ejecutar:
1. ¿Tendrá mejor o peor exactitud que la CNN con pooling (≈95 %)?
2. ¿Y con los dígitos movidos 3 px (la CNN con pooling tenía ≈67 %)?

In [ ]:
inicio = time.time()
entrenar(sin_pool)
print(f"Entrenamiento: {time.time() - inicio:.0f} s")
print(f"                 con pooling   sin pooling")
print(f"Sin mover     :   {exactitud(cnn, X_te):6.1%}       {exactitud(sin_pool, X_te):6.1%}")
movidos = torch.roll(X_te, 3, dims=3)
print(f"Movidos 3 px  :   {exactitud(cnn, movidos):6.1%}       {exactitud(sin_pool, movidos):6.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sin mover, quedan **casi empatadas** (≈95 % la red con pooling y ≈96 % la red sin pooling): con 12 veces más parámetros, la red sin pooling ajusta un poco mejor estos dígitos centrados, a costa de una capa densa enorme.
2. Con los dígitos movidos 3 px, la red **sin pooling** cae a ≈63 %, por debajo de la red con pooling (≈67 %). Su enorme capa densa vuelve a "memorizar posiciones", como la red densa.

Conclusión: el pooling no siempre da más exactitud en los datos "fáciles", pero da **una red más pequeña, más rápida y más tolerante a desplazamientos**.
</details>

**Paso 3 ✏️ — Tu propio diseño.** Diseña una CNN con **tres** bloques conv+pool (por ejemplo 8 → 16 → 32 filtros) que tenga **menos de 10,000 parámetros**. Primero calcula a mano el tamaño del mapa tras el tercer pooling, luego completa `TAM_FINAL` y ejecuta.

In [ ]:
TAM_FINAL = 3      # ✏️ tamaño (alto = ancho) del mapa tras 3 poolings: 28 → 14 → 7 → ?

torch.manual_seed(0)
tres_bloques = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * TAM_FINAL * TAM_FINAL, 10))
print(f"Parámetros: {contar(tres_bloques):,}")
inicio = time.time(); entrenar(tres_bloques)
print(f"Exactitud: {exactitud(tres_bloques, X_te):.1%} · movidos 3 px: {exactitud(tres_bloques, movidos):.1%} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

28 → 14 → 7 → **3** (se redondea hacia abajo). La red tiene unos **8,800 parámetros** (menos de la mitad que la de 2 bloques) y logra ≈92 % de exactitud, un poco menos que la de 2 bloques con 3 épocas. Con los dígitos movidos 3 px resiste **mejor** que la de 2 bloques (≈74 % contra ≈67 %), porque el tercer pooling añade tolerancia y su campo receptivo es mayor. Es un buen trato: menos de la mitad de parámetros y más tolerancia a los desplazamientos, a cambio de un poco de exactitud (con más épocas la diferencia se reduce). Apilar más capas para "ver" zonas más grandes con pocos parámetros es la idea que llevó a las redes muy profundas de la Lección 10.4.
</details>

**Paso 4 ✋ — Explícaselo a alguien.** En tres frases, sin tecnicismos, explica a un amigo por qué la red con pooling resiste mejor que un dígito se mueva.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algo como: *"La red primero busca trazitos pequeños en toda la imagen con las mismas 'lupas'. Luego, en vez de anotar la posición exacta de cada trazo, solo anota 'en esta zona hubo un trazo vertical', y repite eso varias veces con zonas cada vez más grandes. Así, si el dígito se corre un poco, las zonas siguen diciendo casi lo mismo y la respuesta final no cambia."*
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *CNN Explainer* (poloclub.github.io/cnn-explainer): mira el max pooling y los mapas de una CNN real.
- 📘 **Lectura:** Zeiler y Fergus (2013), *Visualizing and Understanding Convolutional Networks*: el artículo que mostró la jerarquía bordes → texturas → partes → objetos.
- 📘 **Lectura:** Distill, *Feature Visualization* (Olah et al., 2017), en inglés, con imágenes de lo que "busca" cada neurona.

---
⬅️ **Anterior:** [10.2 Convoluciones: filtros que detectan patrones](10.2-convoluciones.ipynb) · ➡️ **Siguiente:** [10.4 Arquitecturas icónicas: LeNet, AlexNet, VGG, ResNet](10.4-arquitecturas-iconicas.ipynb)